In [1]:
import tensorflow_datasets as tfds
import tensorflow as tf
import io

In [2]:
imdb, info = tfds.load("imdb_reviews", with_info=True, as_supervised=True, data_dir="./data", download=False)

In [3]:
print(info)

tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='data\\imdb_reviews\\plain_text\\1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
    supervised_keys=('text', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <SplitInfo num_examples=25000, num

In [4]:
train ,test = imdb["train"], imdb["test"]

In [32]:
VOCAB_SIZE = 10000
MAXLEN = 120
EMBEDING_DIM = 32
PADDING_TYPE = "pre"
TRUNC_TYPE = "post"

In [33]:
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
)
train_reviews = train.map(lambda review, label: review)
train_labels = train.map(lambda review, label: label)

test_reviews = test.map(lambda review, label: review)
test_labels = test.map(lambda review, label: label)

In [34]:
vectorize_layer.adapt(train_reviews)


In [35]:
def pad_func(sequences):
    sequences = sequences.ragged_batch(batch_size=sequences.cardinality())
    sequences = sequences.get_single_element()
    sequences = tf.keras.utils.pad_sequences(sequences.numpy(), maxlen=MAXLEN, padding=PADDING_TYPE, truncating=TRUNC_TYPE)
    padded_sequences = tf.data.Dataset.from_tensor_slices(sequences)
    return padded_sequences
    

In [36]:
train_sequences = train_reviews.map(lambda text: vectorize_layer(text)).apply(pad_func)
test_sequences = test_reviews.map(lambda text: vectorize_layer(text)).apply(pad_func)

In [37]:
for ex in train_sequences.take(1):
    print(ex)

tf.Tensor(
[   0    0    0    0   11   14   34  412  384   18   90   28    1    8
   33 1322 3560   42  487    1  191   24   85  152   19   11  217  316
   28   65  240  214    8  489   54   65   85  112   96   22 5596   11
   93  642  743   11   18    7   34  394 9522  170 2464  408    2   88
 1216  137   66  144   51    2    1 7558   66  245   65 2870   16    1
 2860    1    1 1426 5050    3   40    1 1579   17 3560   14  158   19
    4 1216  891 8040    8    4   18   12   14 4059    5   99  146 1241
   10  237  704   12   48   24   93   39   11 7339  152   39 1322    1
   50  398   10   96 1155  851  141    9], shape=(120,), dtype=int32)


In [38]:
train_data_vect = tf.data.Dataset.zip((train_sequences, train_labels))
test_data_vect = tf.data.Dataset.zip((test_sequences, test_labels))

In [39]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(MAXLEN,)),
    tf.keras.layers.Embedding(VOCAB_SIZE, EMBEDING_DIM),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(6, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")

])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 120, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 3840)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 6)              │        23,046 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │             7 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 343,053 (1.31 MB)

 Trainable params: 343,053 (1.31 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
train_data_vect = train_data_vect.batch(32)
test_data_vect = test_data_vect.batch(32)

history = model.fit(train_data_vect, epochs=10, validation_data=test_data_vect)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.7516 - loss: 0.4825 - val_accuracy: 0.8246 - val_loss: 0.3856
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9224 - loss: 0.2110 - val_accuracy: 0.8093 - val_loss: 0.4547
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9858 - loss: 0.0612 - val_accuracy: 0.8058 - val_loss: 0.5534
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9982 - loss: 0.0120 - val_accuracy: 0.8054 - val_loss: 0.6254
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9997 - loss: 0.0030 - val_accuracy: 0.7970 - val_loss: 0.7078
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9998 - loss: 0.0016 - val_accuracy: 0.8027 - val_loss: 0.7617
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 1.0000 - loss: 5.8268e-04 - val_accuracy: 0.8048 - val_loss: 0.7887
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 1.0000 - loss: 2.4610e-04 - val_accu

In [41]:
embedding_layers = model.layers[0]
weights = embedding_layers.get_weights()[0]
print(weights.shape)

(10000, 32)


In [42]:
# Open writeable files
out_v = io.open('vecs.tsv', 'w', encoding='utf-8')
out_m = io.open('meta.tsv', 'w', encoding='utf-8')

# Get the word list
vocabulary = vectorize_layer.get_vocabulary()

# Initialize the loop. Start counting at `1` because `0` is just for the padding
for word_num in range(1, len(vocabulary)):

  # Get the word associated with the current index
  word_name = vocabulary[word_num]

  # Get the embedding weights associated with the current index
  word_embedding = weights[word_num]

  # Write the word name
  out_m.write(word_name + "\n")

  # Write the word embedding
  out_v.write('\t'.join([str(x) for x in word_embedding]) + "\n")

# Close the files
out_v.close()
out_m.close()